# 第 9 章:监督微调(SFT)—— 教模型扮演 assistant

前 8 章我们造了一台能「续写文本」的引擎(预训练),但这台引擎只会**续写** —— 你输入「今天天气」,它接「真好,阳光明媚」也可能接「预报说有雨」。它不知道你在**问问题**,更不知道该以 **assistant 的身份**回答。

**监督微调(Supervised Fine-Tuning, SFT)** 就是教预训练模型「对话」:给定用户的问题,生成 assistant 的回复。

本章对应 minimind 的核心代码:
- `dataset/lm_dataset.py:58-119`(`SFTDataset` 类 + `generate_labels`)
- `trainer/train_full_sft.py`(训练脚本)

> ⭐ **本章核心**是 **answer-only loss masking** —— 只让模型从 assistant 的回复中学习,忽略 prompt 部分。这是 SFT 与预训练最本质的区别。

## 环境准备

导入 minimind 的 tokenizer 和 SFTDataset,加载一个小样本看看 SFT 数据长什么样。

In [ ]:
import sys, json
sys.path.insert(0, '/home/minimind')

from transformers import AutoTokenizer
from dataset.lm_dataset import SFTDataset

# 加载 minimind 的 tokenizer
tokenizer = AutoTokenizer.from_pretrained('/home/minimind/model')

print(f"bos_token: {tokenizer.bos_token!r}  (id={tokenizer.bos_token_id})")
print(f"eos_token: {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")
print(f"pad_token: {tokenizer.pad_token!r}  (id={tokenizer.pad_token_id})")
print(f"vocab_size: {tokenizer.vocab_size}")

&nbsp;

---

## 9.1 为什么预训练后还要 SFT

预训练和 SFT 解决**完全不同的问题**:

| | 预训练 (Pretrain) | 监督微调 (SFT) |
|---|---|---|
| **目标** | 学语言规律 | 学对话格式 |
| **数据** | 纯文本(书籍、网页) | 问答对(conversations) |
| **学什么** | P(下一个 token) | P(回复 \| 提问) |
| **loss 来源** | 所有 token | **仅 assistant 的 token** |
| **学习率** | 5e-4 | **1e-5**(低 50 倍) |
| **起点** | 随机初始化 | `from_weight=pretrain` |

预训练模型像一**个读了很多书但不会聊天的人** —— 知识丰富,但不知道「被问到问题时应该回答」。

SFT 用**对话数据**告诉模型:「当输入是 `[user] 你好 [/user]` 时,你应该输出 `[assistant] 你好!有什么可以帮助你的?[/assistant]`」。

关键代码在 `train_full_sft.py`:

```python
# train_full_sft.py:103
parser.add_argument('--from_weight', default='pretrain', ...)
#         ↑ 基于预训练权重继续训练,不是从头开始

# train_full_sft.py:90
parser.add_argument("--learning_rate", type=float, default=1e-5, ...)
#         ↑ 比预训练(5e-4)低 50 倍 —— 因为模型已经会语言了,只需微调

# train_full_sft.py:136
train_ds = SFTDataset(args.data_path, tokenizer, max_length=args.max_seq_len)
#         ↑ 用 SFTDataset(不是 PretrainDataset)
```

&nbsp;

---

## 9.2 SFT 数据格式

SFT 数据是一组**多轮对话**,每条数据的结构是:

```json
{
  "conversations": [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好!有什么可以帮助你的?"}
  ]
}
```

- `role` 可以是 `system` / `user` / `assistant`
- 多轮对话就是交替的 user → assistant → user → assistant ...

`SFTDataset.__init__` 用 HuggingFace 的 `load_dataset` 读取 JSONL:

```python
# lm_dataset.py:63-64
features = Features({
    'conversations': [{'role': Value('string'), 'content': Value('string'), ...}]
})
self.samples = load_dataset('json', data_files=jsonl_path, split='train', features=features)
```

下面用代码构造一个假的多轮对话样本:

In [ ]:
# 构造一个假的多轮对话样本(模拟 SFT 数据)
sample_conversations = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好!有什么可以帮助你的?"},
    {"role": "user", "content": "1+1等于几?"},
    {"role": "assistant", "content": "1+1等于2。"},
]

print("=== 原始对话数据 ===")
for msg in sample_conversations:
    print(f"  [{msg['role']:9s}] {msg['content']}")

print(f"\n对话轮数: {len(sample_conversations)} 条消息")
print(f"user 轮数:     {sum(1 for m in sample_conversations if m['role']=='user')}")
print(f"assistant 轮数: {sum(1 for m in sample_conversations if m['role']=='assistant')}")

&nbsp;

---

## 9.3 chat_template 渲染

对话数据是结构化的 JSON,但模型只认 **token 序列**。需要把 `{role, content}` 列表渲染成一串文本 —— 这就是 `chat_template` 的作用。

minimind 用 HuggingFace 的 `apply_chat_template` 方法:

```python
# lm_dataset.py:81-86
prompt = self.tokenizer.apply_chat_template(
    messages,
    tokenize=False,          # 先返回字符串,不直接 tokenize
    add_generation_prompt=False,
    tools=tools
)
```

渲染后的文本长这样(Qwen 风格的 ChatML 格式):

```
<bos><|im_start|>user
你好<|im_end|>
<bos><|im_start|>assistant
你好!有什么可以帮助你的?<|im_end|>
```

每个角色用 `<|im_start|>role\n...<|im_end|>` 包裹。`apply_chat_template` 负责拼接这些标记。

In [ ]:
# 用 apply_chat_template 渲染对话
messages = [{"role": "user", "content": "你好"}]

rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("=== apply_chat_template(add_generation_prompt=True) ===")
print(repr(rendered))
print()
print("=== 可读版 ===")
print(rendered.replace('<bos>', '<bos>\n'))

# 渲染完整的 user+assistant 对话
full_messages = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好!有什么可以帮助你的?"},
]
rendered_full = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
print("\n=== 完整对话渲染 ===")
print(rendered_full.replace('<bos>', '<bos>\n'))

注意 `add_generation_prompt` 的区别:
- `=True`: 末尾加 `<bos><|im_start|>assistant\n`,**提示模型从这里开始生成**(推理时用)
- `=False`: 不加额外提示,**完整对话**(训练时用)

> minimind 的特殊标记:`<bos>` 是句首(begin of sequence),`<|im_start|>` / `<|im_end|>` 是 ChatML 格式的角色边界标记。`apply_chat_template` 来自 tokenizer 配置,可以在 `tokenizer_config.json` 里查看模板字符串。

&nbsp;

---

## 9.4 ⭐ answer-only loss masking(本章核心!)

这是 SFT 最关键的设计。先看问题:

### 不做 masking 会怎样?

如果把整个对话(含 user 的提问)都当训练目标,模型会学到什么?

```
训练目标: <bos><|im_start|>user\n你好<|im_end|>\n<bos><|im_start|>assistant\n你好!<|im_end|>
                ↑ 模型会学会「生成 user 的提问」! ↑   ↑ 这才是我们想要的 ↑
```

模型会同时学习:
1. ✅ 怎么**回答**问题(assistant 部分)—— 这是我们想要的
2. ❌ 怎么**提出**问题(user 部分)—— 这是我们**不想要**的!

结果:模型可能会自问自答,或者学着模仿用户的口吻。这不是 assistant 该做的事。

### 解决方案:只从 assistant 的 token 学习

`SFTDataset.generate_labels` 的核心逻辑(lm_dataset.py:88-104):

```python
def generate_labels(self, input_ids):
    labels = [-100] * len(input_ids)       # ① 全部初始化为 -100(忽略)
    i = 0
    while i < len(input_ids):
        # ② 找到 '<bos>assistant\n' 标记 —— assistant 回复的开始
        if input_ids[i:i + len(self.bos_id)] == self.bos_id:
            start = i + len(self.bos_id)   #    跳过标记本身
            end = start
            # ③ 找到 '<eos>\n' 标记 —— assistant 回复的结束
            while end < len(input_ids):
                if input_ids[end:end + len(self.eos_id)] == self.eos_id:
                    break
                end += 1
            # ④ 只有 [start, end+eos] 范围内的 token 被保留(label = token_id)
            for j in range(start, min(end + len(self.eos_id), self.max_length)):
                labels[j] = input_ids[j]
            i = end + len(self.eos_id)     #    跳过已处理的 span
        else:
            i += 1
    return labels
```

逻辑拆解:
1. **初始化**:所有 label 设为 `-100`(CE loss 的 `ignore_index`,不参与计算)
2. **扫描**:遍历整个 token 序列,找 `<bos>assistant\n` 标记
3. **标记 span**:从 `<bos>assistant\n` 之后到 `<eos>\n` 之间的 token,label 设为真实 token_id
4. **重复**:多轮对话中有多个 assistant span,全部找出来

效果:
```
token:  <bos><|im_start|>user\n你好<|im_end|>\n<bos><|im_start|>assistant\n你好!<|im_end|>
label:  -100  -100 -100 -100 -100 -100 -100 -100 -100 你  好  !   <|im_end|>
                                                                        ↑ 只有这些参与 loss
```

下面用代码验证这个过程:

In [ ]:
# === 演示 generate_labels 的效果 ===

# 构造一个完整的对话并渲染
messages = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好!有什么可以帮助你的?"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
input_ids = tokenizer(prompt, add_special_tokens=False).input_ids

print("=== 渲染后的文本 ===")
print(prompt)
print(f"\n=== token 序列(共 {len(input_ids)} 个 token) ===")
for i, tid in enumerate(input_ids):
    token_str = tokenizer.decode([tid])
    print(f"  {i:3d}: id={tid:5d}  token={token_str!r}")

现在手动实现一个 naive 版的 `generate_labels`,逐个 token 扫描,直观展示哪些被保留:

In [ ]:
# === naive 版 generate_labels(手动定位 span)===

bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids
eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids

print(f"bos_id (<bos>assistant\n): {bos_id}")
print(f"  解码: {[tokenizer.decode([t]) for t in bos_id]}")
print(f"eos_id (<eos>\\n):          {eos_id}")
print(f"  解码: {[tokenizer.decode([t]) for t in eos_id]}")

# naive 版:逐个位置扫描
labels = [-100] * len(input_ids)
i = 0
while i < len(input_ids):
    if input_ids[i:i + len(bos_id)] == bos_id:
        start = i + len(bos_id)
        end = start
        while end < len(input_ids):
            if input_ids[end:end + len(eos_id)] == eos_id:
                break
            end += 1
        for j in range(start, min(end + len(eos_id), len(input_ids))):
            labels[j] = input_ids[j]
        i = end + len(eos_id)
    else:
        i += 1

# 统计
n_masked = sum(1 for l in labels if l == -100)
n_active = sum(1 for l in labels if l != -100)
print(f"\n=== 结果 ===")
print(f"总 token 数:    {len(input_ids)}")
print(f"被 mask (-100): {n_masked}  (user/system 部分)")
print(f"参与 loss:      {n_active}  (assistant 部分)")

现在做**可视化** —— 用颜色标注哪些 token 参与 loss(绿),哪些被忽略(灰):

In [ ]:
# === loss masking 可视化 ===
# 绿色 = 参与 loss (assistant token)
# 灰色 = 被 mask (-100, user/system token)

def visualize_loss_mask(input_ids, labels, tokenizer):
    GREEN = '\033[92m'
    GRAY = '\033[90m'
    RESET = '\033[0m'
    
    print(f"{'idx':>3}  {'token_id':>8}  {'token':<16}  {'label':>8}  状态")
    print("-" * 60)
    for i, (tid, lab) in enumerate(zip(input_ids, labels)):
        token_str = tokenizer.decode([tid]).replace('\n', '\\n')
        if lab != -100:
            print(f"{GREEN}{i:3d}  {tid:8d}  {token_str:<16}  {lab:8d}  ✓ 参与 loss{RESET}")
        else:
            print(f"{GRAY}{i:3d}  {tid:8d}  {token_str:<16}  {'-100':>8}  ✗ masked{RESET}")

visualize_loss_mask(input_ids, labels, tokenizer)

看!只有 `<bos>assistant\n` 之后、`<eos>\n` 及之前的内容(label ≠ -100)参与 loss。user 的提问、`<|im_start|>` 标记、role 标签全部被 mask 掉。

### 多轮对话的情况

多轮对话中有**多个** assistant span,`generate_labels` 会把它们全部找出来:

In [ ]:
# === 多轮对话的 loss masking ===

multi_messages = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好!"},
    {"role": "user", "content": "1+1=?"},
    {"role": "assistant", "content": "1+1=2。"},
]
multi_prompt = tokenizer.apply_chat_template(multi_messages, tokenize=False, add_generation_prompt=False)
multi_ids = tokenizer(multi_prompt, add_special_tokens=False).input_ids

# 用相同的逻辑生成 labels
labels2 = [-100] * len(multi_ids)
i = 0
while i < len(multi_ids):
    if multi_ids[i:i + len(bos_id)] == bos_id:
        start = i + len(bos_id)
        end = start
        while end < len(multi_ids):
            if multi_ids[end:end + len(eos_id)] == eos_id:
                break
            end += 1
        for j in range(start, min(end + len(eos_id), len(multi_ids))):
            labels2[j] = multi_ids[j]
        i = end + len(eos_id)
    else:
        i += 1

print(f"多轮对话: {len(multi_messages)} 条消息, {len(multi_ids)} 个 token")
print(f"参与 loss 的 token: {sum(1 for l in labels2 if l != -100)} 个")
print(f"被 mask 的 token:   {sum(1 for l in labels2 if l == -100)} 个")
print()
visualize_loss_mask(multi_ids, labels2, tokenizer)

可以看到**两个 assistant span** 都被正确标记了(两组绿色的 token)。user 部分全部灰色(masked)。

### compact 版

minimind 源码里的实现其实就是我们上面写的逻辑 —— 但它更紧凑,直接放在 `SFTDataset` 类里。核心区别只是代码组织方式:

```python
# lm_dataset.py:65-66 —— 预计算标记的 token id
self.bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids
self.eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids
```

```python
# lm_dataset.py:106-119 —— __getitem__ 里的调用
def __getitem__(self, index):
    sample = self.samples[index]
    conversations = pre_processing_chat(sample['conversations'])  # 预处理
    prompt = self.create_chat_prompt(conversations)                # 渲染 chat_template
    prompt = post_processing_chat(prompt)                          # 后处理
    input_ids = self.tokenizer(prompt).input_ids[:self.max_length] # tokenize + 截断
    input_ids += [self.tokenizer.pad_token_id] * (self.max_length - len(input_ids))  # padding
    labels = self.generate_labels(input_ids)                       # ⭐ loss masking
    return torch.tensor(input_ids), torch.tensor(labels)
```

> **为什么用 `<bos>assistant\n` 而不是 `<|im_start|>assistant`?** 因为 minimind 的 tokenizer 把 `<bos>` 和 `assistant\n` 一起编码,形成固定的 token 序列。用这个序列作为锚点,能精确定位每个 assistant 回复的开始位置。

&nbsp;

---

## 9.5 pre/post processing

`SFTDataset.__getitem__` 在 tokenize 之前有两个预处理步骤:

### pre_processing_chat(20% 概率注入 system prompt)

```python
# lm_dataset.py:9-29
def pre_processing_chat(conversations, add_system_ratio=0.2):
    # tool use 数据完整保留不做处理
    if any(conv.get('tools') for conv in conversations): return conversations

    SYSTEM_PROMPTS = [
        "你是一个知识丰富的AI，尽力为用户提供准确的信息。",
        "你是minimind，一个小巧但有用的语言模型。",
        # ... 共 10 条
    ]
    # 概率性添加 system
    if conversations[0].get('role') != 'system':
        if random.random() < add_system_ratio:  # 20% 概率
            return [{'role': 'system', 'content': random.choice(SYSTEM_PROMPTS)}] + conversations
    return conversations
```

**为什么是 20% 而不是 100%?** 如果每条数据都加 system prompt,模型会**过度依赖**它 —— 推理时如果不加 system,表现会变差。20% 的比例让模型既能处理有 system 的情况,也能处理没有的情况。

### post_processing_chat(80% 概率去掉空 think 标签)

```python
# lm_dataset.py:31-35
def post_processing_chat(prompt_content, empty_think_ratio=0.2):
    # 以 80% 概率移除空思考标签
    if '<think>\n\n</think>\n\n' in prompt_content and random.random() > empty_think_ratio:
        prompt_content = prompt_content.replace('<think>\n\n</think>\n\n', '')
    return prompt_content
```

有些 assistant 回复包含空的思考标签 `<think>\n\n</think>\n\n`(没有实际思考内容)。80% 的情况下会把它去掉,让模型学会直接回答;20% 保留,让模型学会输出空的 think 结构(为后面的推理/思维链训练做准备)。

下面用代码演示这两个预处理的效果:

In [ ]:
import random
random.seed(42)

# === pre_processing_chat 演示 ===
from dataset.lm_dataset import pre_processing_chat, post_processing_chat

conversations = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好!"},
]

print("=== pre_processing_chat(20% 加 system) ===")
counts = {"加了 system": 0, "没加 system": 0}
for _ in range(1000):
    result = pre_processing_chat(conversations)
    if result[0].get('role') == 'system':
        counts["加了 system"] += 1
    else:
        counts["没加 system"] += 1
print(f"1000 次采样: {counts}")
print(f"加 system 的比例: {counts['加了 system']/10:.1f}%")  # ≈ 20%

# === post_processing_chat 演示 ===
print("\n=== post_processing_chat(80% 去空 think) ===")
prompt_with_think = "<bos><|im_start|>assistant\n<think>\n\n</think>\n\n你好!<|im_end|>"

counts2 = {"去掉了 think": 0, "保留了 think": 0}
for _ in range(1000):
    result = post_processing_chat(prompt_with_think)
    if '<think>' not in result:
        counts2["去掉了 think"] += 1
    else:
        counts2["保留了 think"] += 1
print(f"1000 次采样: {counts2}")
print(f"去掉 think 的比例: {counts2['去掉了 think']/10:.1f}%")  # ≈ 80%

&nbsp;

---

## 9.6 SFT vs Pretrain 差异对照

把 `SFTDataset` + `train_full_sft.py` 和 `PretrainDataset` + `train_pretrain.py` 对照来看:

| 维度 | 预训练 (Pretrain) | SFT | 差异说明 |
|---|---|---|---|
| **Dataset 类** | `PretrainDataset` | `SFTDataset` | 不同的数据处理逻辑 |
| **数据格式** | `{"text": "..."}` 纯文本 | `{"conversations": [...]}` 对话 | 结构完全不同 |
| **label 生成** | 所有 token label=token_id | **仅 assistant token**,其余 -100 | ⭐ 核心差异 |
| **学习率** | 5e-4 | **1e-5** | 低 **50 倍** |
| **max_seq_len** | 340 | **768** | SFT 序列更长(对话) |
| **from_weight** | `none`(从头) | `pretrain`(基于预训练) | SFT 是继续训练 |
| **batch_size** | 16 | 16 | 相同 |
| **epochs** | 1 | 2 | SFT 多跑一轮 |

### 为什么学习率低 50 倍?

预训练时模型从零开始,需要大学习率快速学习语言规律。SFT 时模型**已经会语言了**,只需要**微调** —— 学习率太大会把已经学好的知识**冲掉**(catastrophic forgetting)。

1e-5 是一个足够小、能温和调整模型行为、又不会破坏预训练知识的值。

### 为什么 max_seq_len 从 340 变 768?

预训练数据是短文本片段(340 token ≈ 500 字),SFT 数据是**多轮对话**,加上 chat_template 的标记(`<|im_start|>role\n...<|im_end|>`),序列会更长。768 token 能容纳更多轮对话。

In [ ]:
# === 对比 PretrainDataset 和 SFTDataset 的 label 生成 ===

from dataset.lm_dataset import PretrainDataset

# PretrainDataset 的 label 生成逻辑(lm_dataset.py:47-55):
#   labels = input_ids.clone()
#   labels[input_ids == pad_token_id] = -100
# → 只有 padding 被 mask,所有真实 token 都参与 loss!

print("=== PretrainDataset: label 生成 ===")
print("""
tokens = [bos] + text_tokens + [eos] + [pad, pad, pad...]
labels = [bos] + text_tokens + [eos] + [-100, -100, -100...]
         ↑ 所有真实 token 都参与 loss(包括 user 提问!)
""")

print("=== SFTDataset: label 生成 ===")
print("""
tokens = [bos]<|im_start|>user\\n你好<|im_end|>\\n[bos]<|im_start|>assistant\\n你好!<|im_end|>
labels = -100  -100  -100 -100 -100 -100 -100 -100 -100 你 好 ! <|im_end|>
         ↑ user 部分全部 -100(忽略)                        ↑ 只有 assistant 参与 loss
""")

# 数值对比
print("=== 数值对比 ===")
print(f"{'指标':<25} {'Pretrain':>12} {'SFT':>12} {'倍数':>8}")
print("-" * 60)
print(f"{'学习率 (lr)':<25} {'5e-4':>12} {'1e-5':>12} {'1/50':>8}")
print(f"{'max_seq_len':<25} {340:>12} {768:>12} {'2.3x':>8}")
print(f"{'from_weight':<25} {'none':>12} {'pretrain':>12} {'-':>8}")
print(f"{'参与 loss 的 token 比例':<25} {'~100%':>12} {'~30-50%':>12} {'-':>8}")

&nbsp;

---

## 9.7 训练命令与结果

SFT 训练用 `train_full_sft.py`,核心命令:

```bash
cd /home/minimind/trainer

# 单卡训练(默认参数)
python train_full_sft.py

# 自定义参数
python train_full_sft.py \
    --data_path ../dataset/sft_t2t_mini.jsonl \
    --from_weight pretrain \
    --learning_rate 1e-5 \
    --epochs 2 \
    --batch_size 16 \
    --max_seq_len 768 \
    --save_dir ../out \
    --save_weight full_sft
```

训练流程(`train_full_sft.py:84-168`):

```python
# 1. 初始化模型(基于预训练权重)
model, tokenizer = init_model(lm_config, args.from_weight, device=args.device)
#                                        ↑ 'pretrain' → 加载 pretrain_768.pth

# 2. 创建 SFTDataset + DataLoader
train_ds = SFTDataset(args.data_path, tokenizer, max_length=args.max_seq_len)
loader = DataLoader(train_ds, batch_sampler=batch_sampler, ...)

# 3. 优化器
optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate)  # 1e-5

# 4. 训练循环
for epoch in range(args.epochs):  # 2 轮
    for step, (input_ids, labels) in enumerate(loader):
        res = model(input_ids, labels=labels)       # forward(含 CE loss)
        loss = res.loss + res.aux_loss                # MoE aux_loss(密集模型=0)
        loss.backward()                               # 反向传播
        optimizer.step()                              # 更新权重

# 5. 保存权重
# → ../out/full_sft_768.pth
```

训练完成后,权重保存在 `../out/full_sft_768.pth`(命名规则:`{save_weight}_{hidden_size}.pth`)。

In [ ]:
# === 模拟训练循环的核心逻辑(不实际训练)===

import torch
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
model = MiniMindForCausalLM(config)

# 模拟 SFTDataset 返回的一个 batch(batch_size=2, seq_len=64)
batch_input_ids = torch.randint(0, config.vocab_size, (2, 64))
batch_labels = torch.full((2, 64), -100, dtype=torch.long)
# 假设位置 30-40 是 assistant 的 token(参与 loss)
batch_labels[:, 30:41] = batch_input_ids[:, 30:41]

print("=== 模拟 forward pass ===")
print(f"input_ids shape: {batch_input_ids.shape}")  # (2, 64)
print(f"labels shape:    {batch_labels.shape}")      # (2, 64)
print(f"参与 loss 的 token 数: {(batch_labels != -100).sum().item()}")  # 2×11=22
print(f"被 mask 的 token 数:   {(batch_labels == -100).sum().item()}")  # 2×53=106

# forward(计算 loss)
with torch.no_grad():
    output = model(batch_input_ids, labels=batch_labels)
print(f"\nloss: {output.loss.item():.4f}")
print(f"  (随机初始化的模型,loss ≈ ln({config.vocab_size}) = {torch.log(torch.tensor(float(config.vocab_size))).item():.4f})")

print(f"\naux_loss: {output.aux_loss.item()}")  # 0(密集模型)
print(f"logits shape: {output.logits.shape}")   # (2, 64, 6400)

&nbsp;

---

## 9.8 验证:对比 pretrain 和 full_sft 的输出

训练完成后,用 `eval_llm.py` 测试 SFT 模型:

```bash
cd /home/minimind

# 用 SFT 权重测试
python eval_llm.py --weight full_sft

# 对比预训练权重(续写模式)
python eval_llm.py --weight pretrain
```

预期的差异:

| 输入 | pretrain 模型 | full_sft 模型 |
|---|---|---|
| 「你好」 | 续写:「你好,我是...」(可能跑题) | 回答:「你好!有什么可以帮助你的?」 |
| 「1+1等于几」 | 续写:「1+1等于几?这是一个...」 | 回答:「1+1等于2。」 |
| 「写一首诗」 | 续写:可能继续写「写一首诗...」 | 回答:实际写一首诗 |

**pretrain 模型只会续写文本**,因为它学的是 P(下一个 token | 前面所有 token),没有「回答问题」的概念。

**full_sft 模型会回答问题**,因为 SFT 数据教会了它:user 提问后,assistant 应该给出回复。

> `eval_llm.py` 默认用 `--weight full_sft`(line 36),说明 SFT 是 minimind 的标准推理权重。pretrain 权重只是中间产物,不直接用于对话。

In [ ]:
# === 演示 chat_template 在推理时的作用 ===

# 推理时,用 apply_chat_template(add_generation_prompt=True) 构造输入
test_messages = [
    {"role": "user", "content": "1+1等于几?"},
]

# add_generation_prompt=True:末尾加 <bos><|im_start|>assistant\n
inference_prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True  # ⭐ 推理时加这个
)
print("=== 推理时的输入 ===")
print(inference_prompt)
print()
print("模型看到这个输入后,会从 assistant\\n 之后开始生成回复")
print("因为它在 SFT 中学到了:assistant\\n 之后的内容就是回复")

&nbsp;

---

## Summary and takeaways

### 核心要点

1. **SFT = 教预训练模型对话**:预训练学会语言,SFT 学会「回答问题」

2. **answer-only loss masking 是灵魂**:只从 assistant 的 token 学习,忽略 user/system 部分
   ```
   labels = [-100, -100, ..., token_id, token_id, ...]
             ↑ user/prompt(忽略)   ↑ assistant(学这个)
   ```

3. **`generate_labels` 的三步算法**:
   - 全部初始化为 -100
   - 找每个 `<bos>assistant\n` 到 `<eos>\n` 的 span
   - span 内的 label 设为真实 token_id

4. **SFT vs Pretrain 的 3 个关键差异**:
   - 学习率:1e-5 vs 5e-4(低 50 倍,防止灾难性遗忘)
   - label:仅 assistant vs 全部 token
   - 起点:from_weight=pretrain vs 从头开始

5. **pre/post processing 的数据增强**:
   - 20% 注入 system prompt(增强鲁棒性)
   - 80% 去掉空 think 标签(简化回复)

### 流水线总结

```
SFT 数据 (conversations)
    │ pre_processing_chat (20% 加 system prompt)
    ▼
chat_template 渲染 → "<bos><|im_start|>user\n...<|im_end|>\n<bos><|im_start|>assistant\n..."
    │ post_processing_chat (80% 去空 think)
    ▼
tokenize → input_ids [bos, im_start, user, \n, ..., im_end, \n, bos, im_start, assistant, \n, ...]
    │ generate_labels (⭐ answer-only masking)
    ▼
labels     [-100, -100, ..., -100, -100, ..., -100, -100, -100, -100, token_id, token_id, ...]
    │ model(input_ids, labels) → CE loss
    ▼
反向传播 → 更新权重 → full_sft_768.pth
```

---

> **下一步**:SFT 让模型学会了对话,但它的回复质量取决于训练数据。如果想进一步提升回复质量(让模型更「礼貌」、更「有用」),需要 **RLHF/DPO** —— 第 10 章。
>
> → [第 10 章 · DPO 直接偏好优化](../ch10/01_main-chapter-code/README.md)